# Stage 1 — Dataset Audit Runbook

Colab entry point for Stage 1 (`CLAUDE.md` roadmap). Downloads both datasets, runs the FoR-2sec
full audit, the codec check, the DEEP-VOICE **metadata-only** audit (decision D6) and the
cross-dataset overlap check, and archives every output to Drive. These outputs are the evidence
behind `docs/dataset_audit.md`.

**Paths:** every data path comes from `configs/audit.yaml`. The only constants in this notebook
are the bootstrap values in cell 1 (repo URL, branch, clone directory, Drive mount point), which
are needed before the config can be read.

**Re-running:** cells 6 (codec check) and 8 (overlap check) also work from the Drive archive
alone, so they can be re-run in a fresh session without re-downloading anything. Run cell 1 first.

This notebook only runs the audit tools in `src/audit/`. It does no preprocessing, modelling,
training or evaluation.

## 1. Setup: Drive, repo, install, tests

Mounts Google Drive, clones the repo (or fetches and checks out `BRANCH` if already cloned),
prints the Python version and the exact commit, installs `requirements.txt`, and runs the full
test suite with its output printed. **Stops if any test fails**: an audit run on top of a broken
tool is not trustworthy evidence.

Set `BRANCH` to the branch you want to run (default `main`). Record the printed commit hash in
`docs/dataset_audit.md`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

from google.colab import drive

# Bootstrap constants (needed before configs/audit.yaml can be read)
REPO_URL = "https://github.com/Doniaa00/voice-spoofing-detection.git"
BRANCH = "main"
REPO_DIR = Path("/content/voice-spoofing-detection")
DRIVE_MOUNT = "/content/drive"

drive.mount(DRIVE_MOUNT)

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True,
                        check=True).stdout.strip()
print("Python:", sys.version.split()[0])
print("Branch:", BRANCH)
print("Commit:", COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

test = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(test.stdout[-3000:])
print(test.stderr[-3000:])
if test.returncode != 0:
    raise RuntimeError("pytest failed: fix before running the audit. Stopping.")

# Config + shared helpers for the cells below
import yaml
cfg = yaml.safe_load((REPO_DIR / "configs" / "audit.yaml").read_text(encoding="utf-8"))
RAW = Path(cfg["drive"]["raw_root"])
ARCHIVE = Path(cfg["drive"]["archive_root"])


def out_dir(name: str) -> Path:
    """Local audit output dir if present, else its copy in the Drive archive (cell 9 layout)."""
    local = Path(cfg[name]["out"])
    if local.exists():
        return local
    archived = ARCHIVE / local.name
    if not archived.exists():
        raise FileNotFoundError(f"No outputs for {name}: neither {local} nor {archived} exists.")
    print(f"[{name}] local outputs missing, reading the Drive archive: {archived}")
    return archived


def for2sec_revision() -> str:
    """Pinned FoR revision (for2sec.revision in configs/audit.yaml). Required."""
    rev = (cfg["for2sec"].get("revision") or "").strip()
    if not rev:
        raise ValueError("for2sec.revision is empty in configs/audit.yaml: pin the HF commit sha.")
    return rev

## 2. Download FoR-2sec (exact revision)

Downloads the **pinned** Hugging Face revision (`for2sec.revision` in the config) to **local disk**
(`for2sec.root`), not to Drive. **Stops if `for2sec.revision` is empty**: the audit is only valid
for an exact, recorded revision.

Prints the folder/label counts and the non-audio file types. **Stops if no audio is found**
(e.g. a parquet-only snapshot). Then backs the dataset up to Drive `raw/` as **one** `.tar` plus
`for_2sec_REVISION.txt` (one archive copies far faster than 17k small files).

In [ ]:
from collections import Counter
from huggingface_hub import snapshot_download

REPO_ID = "UncovAI/FOR-2sec"
sha = for2sec_revision()  # raises if for2sec.revision is empty
print(f"FoR-2sec revision (pinned HF commit sha): {sha}")

local_for2sec = Path(cfg["for2sec"]["root"])
snapshot_download(repo_id=REPO_ID, repo_type="dataset", revision=sha,
                  local_dir=str(local_for2sec))

audio_exts = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus"}
audio = [p for p in local_for2sec.rglob("*") if p.suffix.lower() in audio_exts]
if not audio:
    raise RuntimeError(f"No audio files under {local_for2sec} for revision {sha} "
                       "(parquet-only snapshot?). Stopping.")
print(f"Found {len(audio)} audio files")
for folder, n in sorted(Counter(str(p.parent.relative_to(local_for2sec)) for p in audio).items()):
    print(f"  {folder}: {n}")
other = Counter(p.suffix.lower() for p in local_for2sec.rglob("*")
                if p.is_file() and p.suffix.lower() not in audio_exts)
print("Non-audio file types:", dict(other))

RAW.mkdir(parents=True, exist_ok=True)
tar_path = RAW / f"for_2sec_{sha[:8]}.tar"
subprocess.run(["tar", "-cf", str(tar_path), "-C", str(local_for2sec.parent), local_for2sec.name],
               check=True)
(RAW / "for_2sec_REVISION.txt").write_text(sha + "\n")
print("Backup:", tar_path, f"({tar_path.stat().st_size / 1e9:.2f} GB)")

## 3. Download DEEP-VOICE (external test set)

> **Hard rule 1 / decisions D6, D20.** DEEP-VOICE is the frozen external test set. **Never open,
> play or plot these files**, including by clicking them in Colab's file browser. Before the
> external run, the only permitted accesses are (D20): download, zip hash, unzip, listing
> paths/extensions/sizes/byte hashes, and `src/audit/deepvoice_metadata_audit.py` (cell 7).
> Nothing reads audio content. This cell only downloads, hashes the zip, unzips, and lists folder
> names and file extensions.

Steps:
1. Installs/upgrades the Kaggle client. The API token is read with `getpass` into the
   `KAGGLE_API_TOKEN` environment variable for this session only. It is never printed or saved.
2. Downloads the zip to local disk (next to `deepvoice.root`).
3. Computes a streamed `sha256sum` of the zip **before** unzipping and **stops if it differs from
   `deepvoice.zip_sha256`** in the config: a different archive is a different external test set.
4. Unzips into `deepvoice.root` and prints folders and file types only.
5. Copies the zip and `deep_voice_ZIP_SHA256.txt` to Drive `raw/`.

In [ ]:
import zipfile
from collections import Counter
from getpass import getpass

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)
os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (hidden): ")

dv_root = Path(cfg["deepvoice"]["root"])
zip_dir = dv_root.parent
zip_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(["kaggle", "datasets", "download", "-d",
                "birdy654/deep-voice-deepfake-voice-recognition", "-p", str(zip_dir)], check=True)
zip_path = zip_dir / "deep-voice-deepfake-voice-recognition.zip"

zsha = subprocess.run(["sha256sum", str(zip_path)], capture_output=True, text=True,
                      check=True).stdout.split()[0]
print("Zip:", zip_path.name, f"{zip_path.stat().st_size / 1e9:.2f} GB", "sha256:", zsha)
expected = (cfg["deepvoice"].get("zip_sha256") or "").strip()
if not expected:
    raise ValueError("deepvoice.zip_sha256 is empty in configs/audit.yaml. Stopping.")
if zsha != expected:
    raise RuntimeError(f"Zip sha256 mismatch: got {zsha}, pinned {expected}. "
                       "The Kaggle archive changed. Do not unzip or audit it. Stopping.")
print("Zip sha256 matches the pinned value.")

dv_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(dv_root)

# Structure only: folder names and extensions. No file is opened.
files = [p for p in dv_root.rglob("*") if p.is_file()]
print("Files per folder:", dict(Counter(str(p.parent.relative_to(dv_root)) for p in files)))
print("File types:", dict(Counter(p.suffix.lower() for p in files)))

RAW.mkdir(parents=True, exist_ok=True)
subprocess.run(["cp", str(zip_path), str(RAW)], check=True)
(RAW / "deep_voice_ZIP_SHA256.txt").write_text(zsha + "\n")
print("Backup saved to Drive:", RAW / zip_path.name)

## 4. Dry run: FoR-2sec audit on the first 500 files

Runs the full pipeline on 500 files so bugs or packaging surprises show up in seconds, and
estimates the full-run time.

**Note:** files are processed in sorted path order, so the first 500 files are **all FAKE**
(`FAKE/` sorts before `REAL/`). Label-based statistics in the dry-run output (shortcut AUCs,
by-label tables) are therefore meaningless; use it only as a smoke test and timing estimate.
Its output goes to `<for2sec.out>_dryrun` and is not archived.

In [ ]:
import time

t = time.time()
r = subprocess.run([sys.executable, "-m", "src.audit.for2sec_audit",
                    "--root", cfg["for2sec"]["root"],
                    "--out", cfg["for2sec"]["out"] + "_dryrun",
                    "--revision", for2sec_revision(), "--limit", "500"],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError("Dry run failed. Stopping.")
elapsed = time.time() - t
n_files = sum(1 for p in Path(cfg["for2sec"]["root"]).rglob("*.wav"))
print(f"Dry run: {elapsed:.0f}s -> estimated full run ~{elapsed / 500 * n_files / 60:.0f} min "
      f"for {n_files} files")

## 5. Full FoR-2sec audit

Same command without `--limit`. Writes the per-file inventory, duplicate and near-duplicate
tables (including the D14 threshold selection), the shortcut screens, and the machine summary
that `docs/dataset_audit.md` §2–§4 draws on. Prints `for2sec_summary.md` at the end.

In [ ]:
import time

t = time.time()
r = subprocess.run([sys.executable, "-m", "src.audit.for2sec_audit",
                    "--root", cfg["for2sec"]["root"],
                    "--out", cfg["for2sec"]["out"],
                    "--revision", for2sec_revision()],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError("FoR audit failed. Stopping.")
print(f"Full run: {(time.time() - t) / 60:.1f} min\n")
print((Path(cfg["for2sec"]["out"]) / "for2sec_summary.md").read_text(encoding="utf-8"))

## 6. Codec check (FoR-2sec): REAL / FAKE_mp3 / FAKE_other

Follow-up to the shortcut screen: 7,592 FAKE filenames carry an `.mp3` step, and no REAL ones do.
This splits FAKE into **FAKE_mp3** (stem contains `mp3`) and **FAKE_other**, and compares each
with REAL: counts, medians, and univariate separability (AUC, 0.5 = none). Evidence for
`docs/dataset_audit.md` §3 and decision D16.

Reads `for2sec_inventory.csv` from the local output dir, or from the Drive archive if the local
dir does not exist. Saves `for2sec_codec_check.csv` (one row per group) and
`for2sec_codec_check.json` next to that inventory.

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd
from src.audit.common import auc_separability

for_out = out_dir("for2sec")
inv = pd.read_csv(for_out / "for2sec_inventory.csv")
is_mp3 = inv["stem"].str.lower().str.contains("mp3")
inv["group"] = "REAL"
inv.loc[(inv.label == "FAKE") & is_mp3, "group"] = "FAKE_mp3"
inv.loc[(inv.label == "FAKE") & ~is_mp3, "group"] = "FAKE_other"
real_mp3 = int(((inv.label == "REAL") & is_mp3).sum())
print(inv["group"].value_counts(), "\n")
print("REAL clips with 'mp3' in the stem:", real_mp3, "\n")

cols = ["bw99_hz", "rms_dbfs", "silence_frac", "dc_offset", "clip_frac"]
med = inv.groupby("group")[cols].median()
print("Medians by group:\n", med.round(4), "\n")

rows = []
for g in ["REAL", "FAKE_mp3", "FAKE_other"]:
    row = {"group": g, "n": int((inv.group == g).sum())}
    row.update({f"median_{c}": float(med.loc[g, c]) for c in cols})
    sub = inv[inv.group.isin([g, "REAL"])]
    for c in cols:
        row[f"sep_vs_REAL_{c}"] = (float(auc_separability(sub[c], sub["label"]))
                                   if g != "REAL" else None)
    rows.append(row)
codec = pd.DataFrame(rows)
for g in ["FAKE_mp3", "FAKE_other"]:
    r_ = codec.set_index("group").loc[g]
    print(f"{g} vs REAL separability:", {c: round(float(r_[f"sep_vs_REAL_{c}"]), 3) for c in cols})

codec.to_csv(for_out / "for2sec_codec_check.csv", index=False)
(for_out / "for2sec_codec_check.json").write_text(json.dumps({
    "check": "FoR-2sec codec check (REAL / FAKE_mp3 / FAKE_other)",
    "rule": "FAKE_mp3 = FAKE with 'mp3' in the lower-cased file stem; FAKE_other = remaining FAKE",
    "source_inventory": str(for_out / "for2sec_inventory.csv"),
    "revision": for2sec_revision(),
    "code_commit": COMMIT,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "real_with_mp3_in_stem": real_mp3,
    "separability": "max(AUC, 1-AUC) of each group vs REAL (src.audit.common.auc_separability)",
    "groups": rows,
}, indent=2), encoding="utf-8")
print("\nSaved:", for_out / "for2sec_codec_check.csv", "and .json")

## 7. DEEP-VOICE metadata-only audit

Runs `src/audit/deepvoice_metadata_audit.py` on `<deepvoice.root>/<deepvoice.audit_subdir>`
(`KAGGLE/`) only. `DEMONSTRATION/` is excluded and never opened. Reads file headers, byte hashes,
filenames and the CSV schema/label counts. The module blocks audio-decoding APIs while it runs
and raises if anything calls them. The revision string pins the exact zip by its sha256
(`deepvoice.zip_sha256`, checked by cell 3) and records the download date.

Prints `deepvoice_summary.md`, which starts with the blind-audit attestation.

In [ ]:
dv_audit_root = Path(cfg["deepvoice"]["root"]) / cfg["deepvoice"]["audit_subdir"]
zsha = (cfg["deepvoice"].get("zip_sha256") or "").strip()
if not zsha:
    raise ValueError("deepvoice.zip_sha256 is empty in configs/audit.yaml. Stopping.")
dv_revision = f"kaggle zip-sha256:{zsha}, downloaded 2026-10-04"

r = subprocess.run([sys.executable, "-m", "src.audit.deepvoice_metadata_audit",
                    "--root", str(dv_audit_root),
                    "--out", cfg["deepvoice"]["out"],
                    "--revision", dv_revision],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError("DEEP-VOICE metadata audit failed. Stopping.")
print((Path(cfg["deepvoice"]["out"]) / "deepvoice_summary.md").read_text(encoding="utf-8"))

## 8. FoR ∩ DEEP-VOICE byte-hash overlap

Compares `sha256_bytes` between the two audit inventories. **No DEEP-VOICE file is read here**:
its hashes come from the inventory written by the metadata audit (cell 7). The expected overlap
is 0. A non-zero count would mean identical files in the development pool and the external test
set. That would be reported in `docs/dataset_audit.md` §4–§5, never fixed silently.

Reads both inventories locally, or from the Drive archive if the local dirs do not exist. Saves
the result to `<archive_root>/cross_dataset_overlap.json`.

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd

for_out, dv_out = out_dir("for2sec"), out_dir("deepvoice")
for_hashes = set(pd.read_csv(for_out / "for2sec_inventory.csv")["sha256_bytes"].dropna())
dv_hashes = set(pd.read_csv(dv_out / "deepvoice_inventory.csv")["sha256_bytes"].dropna())
overlap = sorted(for_hashes & dv_hashes)
print(f"FoR files hashed: {len(for_hashes)} · DEEP-VOICE files hashed: {len(dv_hashes)}")
print(f"Identical files shared by FoR and DEEP-VOICE: {len(overlap)} (expected 0)")
for h in overlap:
    print("  ", h)

ARCHIVE.mkdir(parents=True, exist_ok=True)
(ARCHIVE / "cross_dataset_overlap.json").write_text(json.dumps({
    "check": "FoR-2sec ∩ DEEP-VOICE identical files (sha256 of file bytes)",
    "for2sec_inventory": str(for_out / "for2sec_inventory.csv"),
    "deepvoice_inventory": str(dv_out / "deepvoice_inventory.csv"),
    "for2sec_revision": for2sec_revision(),
    "code_commit": COMMIT,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "n_for2sec_hashes": len(for_hashes),
    "n_deepvoice_hashes": len(dv_hashes),
    "n_overlap": len(overlap),
    "overlapping_sha256": overlap,
}, indent=2), encoding="utf-8")
print("Saved:", ARCHIVE / "cross_dataset_overlap.json")

## 9. Archive outputs to Drive

Copies both local output dirs to `<archive_root>/<dir name>` (`for2sec/`, `deepvoice/`), the
layout cells 6 and 8 read from in later sessions. This includes the codec-check files written
into the FoR output dir. A dir that does not exist locally (e.g. a re-run from the archive) is
skipped: its archived copy is already the latest.

In [ ]:
import shutil

ARCHIVE.mkdir(parents=True, exist_ok=True)
for name in ("for2sec", "deepvoice"):
    src = Path(cfg[name]["out"])
    if not src.exists():
        print(f"[{name}] no local outputs at {src}; archive left as is")
        continue
    dst = ARCHIVE / src.name
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Copied {src} -> {dst}")
print("Archive contents:", sorted(p.name for p in ARCHIVE.iterdir()))